# 实践 13.3  随机划分与整家系留出对模型评价的影响

《统计遗传与智能育种学》第十三章 · 电子资源 `13c_family_leakage.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：13c_family_leakage.ipynb。实践模拟24个家系×18个个体，每个家系共享祖先基因型背景，性状由全基因组加性效应、独立家系背景效应和残差构成。在相同Ridge/MLP设置下，代码比较随机6折与整家系6折，两种方案都用OOF预测覆盖全432个样本，并汇总各折测试个体与当折训练样本的最大余弦相似性（基因型按训练折标准化）。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：SEED=1303；432个样本、24家系、220标记。随机6折的平均最大基因组相似性=0.841，Ridge OOF r=0.664、RMSE=1.267，MLP OOF r=0.633、RMSE=1.324；整家系6折的对应相似性降到0.181，Ridge r=0.290、RMSE=1.843，MLP r=0.036、RMSE=2.335。在本模拟中，整家系留出的预测误差较大。

## 输入、完整代码、实际输出与解释

输入与可复现条件：24个家系×18个个体、220个标记；SEED=1303；性状含加性效应、标准差1.2的独立家系背景效应和残差；无外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1303
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1303


In [2]:
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, PredefinedSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

nfam, per, m = 24, 18, 220
founder=rng.binomial(2,rng.uniform(0.1,0.5,m),size=(nfam,m)).astype(float)
X=[]
fam=[]
for f in range(nfam):
    for _ in range(per):
        z=founder[f].copy()
        flip=rng.random(m)<0.08
        z[flip]=rng.binomial(2,0.5,flip.sum())
        X.append(z)
        fam.append(f)
X=np.asarray(X)
fam=np.asarray(fam)
beta=rng.normal(0,0.1,m)
fam_eff=rng.normal(0,1.2,nfam)
y=X@beta + fam_eff[fam] + rng.normal(0,1,len(X))
print('samples/families/markers',len(X),nfam,m)


samples/families/markers 432 24 220


In [3]:
def make_model(kind, fold):
    if kind == 'Ridge':
        estimator=Ridge(alpha=30)
    elif kind == 'MLP':
        estimator=MLPRegressor(hidden_layer_sizes=(32,),alpha=0.05,max_iter=350,
            early_stopping=True,random_state=SEED+fold)
    else:
        raise ValueError(kind)
    return make_pipeline(StandardScaler(),estimator)

def max_genomic_similarity(train_idx,test_idx):
    mu=X[train_idx].mean(axis=0)
    sd=X[train_idx].std(axis=0)
    sd[sd==0]=1
    A=(X[train_idx]-mu)/sd
    B=(X[test_idx]-mu)/sd
    A/=np.maximum(np.linalg.norm(A,axis=1,keepdims=True),1e-12)
    B/=np.maximum(np.linalg.norm(B,axis=1,keepdims=True),1e-12)
    return float(np.mean(np.max(B@A.T,axis=1)))

schemes={
    'random 6-fold':KFold(n_splits=6,shuffle=True,random_state=SEED),
    'family 6-fold':PredefinedSplit(5-fam%6)
}
for scheme,splitter in schemes.items():
    pred={k:np.full(len(y),np.nan) for k in ['Ridge','MLP']}
    similarities=[]
    splits=splitter.split(X,y)
    for fold,(train_idx,test_idx) in enumerate(splits):
        similarities.append(max_genomic_similarity(train_idx,test_idx))
        for kind in pred:
            model=make_model(kind,fold)
            model.fit(X[train_idx],y[train_idx])
            pred[kind][test_idx]=model.predict(X[test_idx])
    print(scheme,'mean max genomic similarity',round(float(np.mean(similarities)),3))
    for kind in ['Ridge','MLP']:
        r=float(np.corrcoef(pred[kind],y)[0,1])
        rmse=float(np.sqrt(np.mean((pred[kind]-y)**2)))
        print(' ',kind,'OOF r',round(r,3),'RMSE',round(rmse,3))


random 6-fold mean max genomic similarity 0.841
  Ridge OOF r 0.664 RMSE 1.267
  MLP OOF r 0.633 RMSE 1.324
family 6-fold mean max genomic similarity 0.181
  Ridge OOF r 0.29 RMSE 1.843
  MLP OOF r 0.036 RMSE 2.335


结果解释：随机6折下训练与测试个体更相似。整家系划分按家系编号固定外折，MLP早停仍在训练侧随机留出。本模拟还人为加入无法由未见家系直接获知的独立家系背景效应，因此家系外推尤其困难。真实育种数据的差异幅度由亲缘、连锁不平衡（LD）、群体结构和性状遗传架构共同决定。

可修改练习：把家系背景效应的标准差从1.2改为0和0.5，比较两种交叉验证；再把每家系样本数从18降到8，观察随机K折的近亲优势和OOF稳定性如何变化。